In [ ]:
import xarray as xr
from pathlib import Path
from itertools import product
from datetime import datetime

from utils import make_kernel_filename


def make_kernel_path_alb(year, month, band="sw", sky="cld"):
    assert band in ["sw", "lw"], "band must be 'sw' or 'lw'"
    assert sky in ["cld", "clr"], "sky must be 'cld' or 'clr'"

    return (
        Path(f"{year}")
        / "monthly_kernel"
        / band
        / "alb"
        / f"RRTM_kernel_{sky}_alb_TOA_SFC_{month}.nc"
    )


def make_kernel_path_ts(year, month, band="lw", sky="cld"):
    assert band in ["sw", "lw"], "band must be 'sw' or 'lw'"
    assert sky in ["cld", "clr"], "sky must be 'cld' or 'clr'"

    return (
        Path(f"{year}")
        / "monthly_kernel"
        / band
        / "ts"
        / f"RRTM_kernel_{sky}_ts_TOA_SFC_{month}.nc"
    )


def make_kernel_path_wv(year, month, band="sw", sky="cld"):
    assert band in ["sw", "lw"], "band must be 'sw' or 'lw'"
    assert sky in ["cld", "clr"], "sky must be 'cld' or 'clr'"

    return (
        Path(f"{year}")
        / "monthly_kernel"
        / band
        / "wv_dt_xxtscheme"
        / f"RRTM_kernel_{sky}_wv_TOA_SFC_{month}.nc"
    )


def assign_date_coord(ds: xr.Dataset | xr.DataArray, year, month):
    return ds.assign_coords(date=datetime(int(year), int(month), 1)).expand_dims(
        dim="date"
    )


def select_toa_arr(ds: xr.Dataset | xr.DataArray):
    return ds["TOA"].sel(up_down_net=3, band=1)


def build_albedo_kernel_dataset(base, years, months):
    year_months = list(product(years, months))
    kernel_dataarrays_clr = [
        select_toa_arr(
            assign_date_coord(
                xr.open_dataset(
                    Path(base) / make_kernel_path_alb(year, month, band="sw", sky="clr")
                ),
                year,
                month,
            )
        )
        for year, month in year_months
    ]
    kernel_dataarrays_cld = [
        select_toa_arr(
            assign_date_coord(
                xr.open_dataset(
                    Path(base) / make_kernel_path_alb(year, month, band="sw", sky="cld")
                ),
                year,
                month,
            )
        )
        for year, month in year_months
    ]
    kernel_ds_clr = xr.combine_by_coords(kernel_dataarrays_clr)
    kernel_ds_cld = xr.combine_by_coords(kernel_dataarrays_cld)
    ds = xr.Dataset(
        data_vars=dict(
            fal=(
                ["all_clr", "date", "latitude", "longitude"],
                [kernel_ds_cld.TOA.data, kernel_ds_clr.TOA.data],
            ),
        ),
        coords=dict(
            latitude=("latitude", kernel_ds_cld.latitude.data),
            longitude=("longitude", kernel_ds_clr.longitude.data),
            date=("date", kernel_ds_clr.date.data),
            all_clr=("all_clr", ["all", "clr"]),
        ),
    )
    ds = ds.transpose("date", "latitude", "longitude", "all_clr")
    return ds


def build_skt_kernel_dataset(base, years, months):
    year_months = list(product(years, months))
    cld_arrays = [
        select_toa_arr(
            assign_date_coord(
                xr.open_dataset(
                    Path(base) / make_kernel_path_ts(year, month, band="lw", sky="cld")
                ),
                year,
                month,
            )
        )
        for year, month in year_months
    ]
    clr_arrays = [
        select_toa_arr(
            assign_date_coord(
                xr.open_dataset(
                    Path(base) / make_kernel_path_ts(year, month, band="lw", sky="clr")
                ),
                year,
                month,
            )
        )
        for year, month in year_months
    ]
    kernel_ds_cld = xr.combine_by_coords(cld_arrays)
    kernel_ds_clr = xr.combine_by_coords(clr_arrays)
    kernel_ds_cld = kernel_ds_cld["TOA"]
    kernel_ds_clr = kernel_ds_clr["TOA"]
    ds = xr.Dataset(
        data_vars=dict(
            skt=(
                ["all_clr", "date", "latitude", "longitude"],
                [kernel_ds_cld.data, kernel_ds_clr.data],
            ),
        ),
        coords=dict(
            latitude=("latitude", kernel_ds_cld.latitude.data),
            longitude=("longitude", kernel_ds_clr.longitude.data),
            date=("date", kernel_ds_clr.date.data),
            all_clr=("all_clr", ["all", "clr"]),
        ),
    )
    ds = ds.transpose("date", "latitude", "longitude", "all_clr")
    return ds


def build_wv_kernel_dataset(base, years, months, band="sw"):
    year_months = list(product(years, months))
    cld_arrays = []
    clr_arrays = []

    for year, month in year_months:
        ds_cld = xr.open_dataset(
            Path(base) / make_kernel_path_wv(year, month, band=band, sky="cld")
        )
        ds_clr = xr.open_dataset(
            Path(base) / make_kernel_path_wv(year, month, band=band, sky="clr")
        )

        ds_cld = select_toa_arr(ds_cld)
        ds_clr = select_toa_arr(ds_clr)

        ds_cld = ds_cld.sum(dim='level')
        ds_clr = ds_clr.sum(dim='level')

        ds_cld = assign_date_coord(ds_cld, year, month)
        ds_clr = assign_date_coord(ds_clr, year, month)

        cld_arrays.append(ds_cld)
        clr_arrays.append(ds_clr)

    kernel_ds_cld = xr.combine_by_coords(cld_arrays)
    kernel_ds_clr = xr.combine_by_coords(clr_arrays)

    ds = xr.Dataset(
        data_vars=dict(
            tcwv=(
                ["all_clr", "date", "latitude", "longitude"],
                [kernel_ds_cld.data, kernel_ds_clr.data],
            ),
        ),
        coords=dict(
            latitude=("latitude", kernel_ds_cld.latitude.data),
            longitude=("longitude", kernel_ds_clr.longitude.data),
            date=("date", kernel_ds_clr.date.data),
            all_clr=("all_clr", ["all", "clr"]),
        ),
    )
    ds = ds.transpose("date", "latitude", "longitude", "all_clr")
    return ds


def write_yearly_kernels(ds, output_dir, years, var):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for year in years:
        path = output_dir / make_kernel_filename(year, var)
        kernel_subset = ds.sel(date=(ds.date.dt.year == year))
        kernel_subset.to_netcdf(path)


In [ ]:
base = Path("./data/kernels_shared")
years = [2011, 2012, 2013, 2014, 2015]
months = [f"{m:02d}" for m in range(1,13)]
print(months, years)

In [ ]:
fal_kernel_ds = build_albedo_kernel_dataset(base, years, months)
skt_kernel_ds = build_skt_kernel_dataset(base, years, months)
sw_tcwv_kernel_ds = build_wv_kernel_dataset(base, years, months, band='sw')
lw_tcwv_kernel_ds = build_wv_kernel_dataset(base, years, months, band='lw')

write_yearly_kernels(fal_kernel_ds, "data/fal/kernels", years, 'fal')
write_yearly_kernels(sw_tcwv_kernel_ds, "data/fal/kernels", years, 'tcwv')
write_yearly_kernels(skt_kernel_ds, "data/ts/kernels", years, 'ts')
write_yearly_kernels(lw_tcwv_kernel_ds, "data/ts/kernels", years, 'tcwv')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(3,4, figsize=(24,16))

for i, month in enumerate(range(1, 13)):
    m = f"{month:02d}"

    r = i // 4
    c = i % 4
    (sw_tcwv_kernel_ds.tcwv.sel(date=f'2011-{m}', all_clr='all')[0]).plot.contourf(ax=ax[r][c], levels=32)

plt.show()


In [ ]:
fig, ax = plt.subplots(3,4, figsize=(24,16))

for i, month in enumerate(range(1, 13)):
    m = f"{month:02d}"

    r = i // 4
    c = i % 4
    dtcwv.sel(date=f'2011-{m}')[0].plot.contourf(ax=ax[r][c], levels=32)

plt.show()